# Hydrology
This notebook documents the modelling of hydrological conditions in western Switzerland. It contains four separate models:
- Lake Level Model
- Stream Model
- Topographic Wettness Index
- Bog Model
The modelling approach was necessary because the present hydrological state of the region is strongly shaped by large-scale melioration projects undertaken during the 19th and 20th centuries, particularly around Lake Biel, Lake Neuchâtel, and Lake Murten. The models reconstruct more natural hydrological conditions using Monte Carlo simulation approaches.
None of the hydrological models were used directly as covariates in the final predictive framework. However, they form the basis for movement-, vegetation-, and crop-related parameters.
All models rely on repeated random perturbation of the DEM and are aggregated by cell-wise averaging. Results are stored in separate file geodatabases.
## 1. Lake Level Model
### Input
- Three local DEM: *LakeBiel_DEM*, *lakeMurten_DEM*, and *LakeNeuchatel_DEM*
- Base DEM (for spatial extent and resolution)
### Method
Lake Biel, Lake Murten, and Lake Neuchâtel form a connected hydrological system with different but interdependent lake levels. Therefore, only one reference lake level needs to be defined.
Based on archaeological and ecological evidence, a transgression phase at the end of the Late Bronze Age suggests Early Iron Age lake levels comparable to those of the Early Modern period. A mean lake level of **431.5 m** was therefore assumed for Lake Neuchâtel.
Relative offsets were applied:
- Lake Biel: -0.7
- Lake Murten: +0.5 m
Seasonal fluctuations of ±1.5 m were assumed. During each of 200 Monte Carlo iterations, a normally distributed random value (mean = 0, SD = 1.5 m) was added to the mean lake levels.
For each iteration and lake:
- Cells below the simulated lake level were classified as 1
- Cells above were classified as 0
All rasters were aggregated using cell-wise averaging and mosaicked into a final raster. Results are stored in lakelevels.gdb
## 2. Stream Model
### Input
- DEM
- Morpohological stream type raster (*River_Types*)
- Stream width statistics table (*Stream_Statistics*)
### Method
The model follows a Monte Carlo approach with 200 iterations. In each iteration:
1. The DEM is randomly perturbed (normal distribution, mean = 0, SD = 1.5 m).
2. Streams are derived from the *DeriveStreamAsRaster* tool using an accumulation threshold of 10 square kilometres
3. Flow direction (D8) and flow accumulation are derived.
4. The stream raster is classified according to morphological stream types.
5. Streams are converted to polylines.
6. Width statistics are joined from Stream_Statistics.
Stream widths were derived from near-natural stream characteristics compiled by the Swiss Federal Office for the Environment. Assuming a normal distribution of widths, four certainty levels were calculated:
- 100% (minimum width)
- 75%
- 43%
- 16%
Polylines were buffered using these widths, converted to raster format, and assigned their respective likelihood values.
All iterations were aggregated using cell-wise averaging. Results are stored in *Stream_System.gdb*.
## 3. Topographic Wettness Index Model
### Input
- DEM
### Method
The Topographic Wetness Index (TWI) is defined as *ln(a/tan(β))* where 
- a = upslope contributing area
- β = local slope (in radians)
For each of 200 iterations, the DEM was randomly perturbed (normal distribution, mean = 0, SD = 1.5 m).
- Flow direction (D8) and flow accumulation were calculated.
- Slope was derived in degrees and converted to radians.
- A value of 1 was added to flow accumulation to avoid division by zero.
- TWI was computed and stored.
All iterations were aggregated using cell-wise averaging. Results are stored in *TWI.gdb*.
## 4. Bog Model
### Input
- Lake Level Model output (*Flooding_Raster*)
- TWI raster
- Soil permeability raster (*Permeability*)
- Historical vegetation polygons (*Bog*, *No_bog*)
### Method
A random forest classifier was trained to predict marshy areas under reconstructed hydrological conditions.
For each of 200 iterations:
1. 200 random presence points were sampled within the *Bog* polygon.
2. 200 random absence points were sampled within the *No_bog* polygon.
3. Points were merged into a training dataset.
4. A forest-based model (200 trees) was trained with 20% training proportion, 10 validation runs, and uncertainty estimation enabled
The model predicts the most probable class (marsh vs non-marsh) for each cell.
All predicted rasters were aggregated by cell-wise averaging to create a final mean probability surface. Results are stored in *RF_Bog.gdb*.
Model performance (out-of-bag metrics) indicates strong generalisation (MCC ≈ 0.85; F1 ≈ 0.93), and spatial patterns align well with independent historical maps.

## Implementation instructions
- Add all data under *data/01_Covariate/01.1_Hydrology/Hydrology.gdb* to the current map
- Adjust input manually (number of iterations, lake levels)
- Execute each cell sequentially

In [ ]:
# Lake Level Model
import os
import arcpy
from arcpy.sa import *
from numpy import random

# ----------------------- CONFIGURATION -----------------------
# set parallel cores
arcpy.env.processorType = "CPU"
arcpy.env.addOutputsToMap = False
arcpy.env.parallelProcessingFactor = "100%"

# define digital elevation model
in_raster = r"DEM"

# create empty lists
in_rasters_lakeneuchatel = []
in_rasters_lakebiel = []
in_rasters_lakemurten= []

# configure environment settings
arcpy.env.extent = in_raster
arcpy.env.cellSize = in_raster
arcpy.env.overwriteOutput = True

# create describe object
desc = arcpy.Describe(arcpy.Describe(in_raster).path)
print(desc.path)

# create new file geodatabase and define it as workspace
arcpy.management.CreateFileGDB(desc.path,"lakelevels.gdb")
out_path = os.path.join(desc.path,"lakelevels.gdb")
print(out_path)
arcpy.env.workspace = out_path

# ----------------------- MODELLING -----------------------
#define lake level Lake Neuchâtel, Lake Biel = Lake Neuchâtel-0.7, and Lake Murten = Lake Neuchâtel +0.5
n = 431.5 #<-- define the lake level for lake Neuchâtel
m = n+0.5
b = n-0.7

# Iteration of lake levels, create a loop
i = 200#<-- number of iterations
for x in range (i):
    print (x)
    # create random deviation from mean lake levels
    r = random.normal(loc=0,scale=1.5)
    name = "LakeNeuchatel"+str(x)
    print(r)
    print(name)
    z = n+r
    print(z)
    out_raster = os.path.join(out_path,name)
    outCon = Con(Raster("LakeNeuchatel_DEM")<=z,1,0)
    outCon.save(out_raster)
    in_rasters_lakeneuchatel.append(out_raster)
    
    z = b+r
    name = "LakeBiel"+str(x)
    print(name)
    print(z)
    out_raster = os.path.join(out_path,name)
    outCon = Con(Raster("LakeBiel_DEM")<=z,1,0)
    outCon.save(out_raster)
    in_rasters_lakebiel.append(out_raster)
    
    z = m+r
    name = "LakeMurten"+str(x)
    print(name)
    print(z)
    out_raster = os.path.join(out_path,name)
    outCon = Con(Raster("LakeMurten_DEM")<=z,1,0)
    outCon.save(out_raster)
    in_rasters_lakemurten.append(out_raster)

# ----------------------- AGGREGATION -----------------------
# aggregate all rasters in the lists and divide them by the number of iterations
localsum4315_lakeneuchatel = arcpy.sa.CellStatistics(in_rasters_lakeneuchatel,"SUM","DATA")/i
localsum4315_lakebiel = arcpy.sa.CellStatistics(in_rasters_lakebiel,"SUM","DATA")/i
localsum4315_lakemurten = arcpy.sa.CellStatistics(in_rasters_lakemurten,"SUM","DATA")/i

# save the lake rasters
out_result_lakeneuchatel = os.path.join(out_path,"localsum4315_LakeNeuchatel")
localsum4315_lakeneuchatel.save(out_result_lakeneuchatel)
out_result_lakebiel = os.path.join(out_path,"localsum4315_LakeBiel")
localsum4315_lakebiel.save(out_result_lakebiel)
out_result_lakemurten = os.path.join(out_path,"localsum4315_LakeMurten")
localsum4315_lakemurten.save(out_result_lakemurten)

# Mosaic the seperate lake rasters to new lake model
arcpy.env.workspace = out_path
arcpy.MosaicToNewRaster_management(input_rasters = "localsum4315_LakeBiel;localsum4315_LakeNeuchatel;localsum4315_LakeMurten",output_location = out_path,raster_dataset_name_with_extension = "Lakelevels4315",cellsize = "25",number_of_bands = "1",mosaic_method = "FIRST",mosaic_colormap_mode = "FIRST")

print("completed")

In [ ]:
# Stream Model
import arcpy
import os
from arcpy.sa import *

# ----------------------- CONFIGURATION -----------------------
# set parallel cores
arcpy.env.addOutputsToMap = False
arcpy.env.parallelProcessingFactor = "75%"

# define digital elevation model
in_raster = r"DEM"

# configure environment settings
arcpy.env.extent = in_raster
arcpy.env.cellSize = in_raster
arcpy.env.overwriteOutput = True
arcpy.env.workspace = "memory"
    
# create describe object
desc = arcpy.Describe(arcpy.Describe(in_raster).path)
print(desc.path)

# create new file geodatabase and define it as workspace
arcpy.management.CreateFileGDB(desc.path,"stream_system.gdb")
out_path = os.path.join(desc.path,"stream_system.gdb")
print(out_path)
arcpy.env.workspace = out_path

# ----------------------- MODELLING -----------------------
i = 200 #<-- number of iterations 
# calculate flow accumulation
for x in range(i):
    # define name of streams
    name = "Stream"+str(x)
    print(name)

    # create random raster
    Random_Raster = arcpy.management.CreateRandomRaster(out_path = out_path,
                                                      out_name = "Random_dhm25",
                                                      distributio n= "NORMAL 0.0 1.5",
                                                      cellsize = 25,
                                                      build_rat = "DO_NOT_BUILD")
    
    # add random raster to DEM
    Raster_Random_Raster = Raster(Random_Raster)
    Raster_in_raster = Raster(in_raster)
    output_raster = Raster_Random_Raster + Raster_in_raster
    
    # calculate stream
    out_raster_stream = arcpy.sa.DeriveStreamAsRaster(in_surface_raster = output_raster,
                                                    in_depressions_data = None,
                                                    in_weight_raster = None,
                                                    accumulation_threshold = "10 SquareKilometers",
                                                    stream_designation_method = "CONSTANT",
                                                    force_flow = "NORMAL")
    
    # assign stream to morphological category
    out_stream_raster_class = RasterCalculator([out_raster_stream,"Stream_Types"],["x","y"],"Con(x==1,y,x)")
    
    # make a line-feature from raster - the river type is saved under "grid_code"
    arcpy.ddd.Int(out_stream_raster_class,"Int_out_raster")
    arcpy.conversion.RasterToPolyline(
        in_raster = "Int_out_raster",
        out_polyline_featur  s= "polyline",
        background_value = "ZERO",
        minimum_dangle_length = 0,
        simplify = "SIMPLIFY",
        raster_field = "Value")

    # join the line-feature with the attribute table with the stream widths
    arcpy.management.JoinField(in_dat a= "memory/polyline",
                               in_field = "grid_code",
                               join_table = "Stream_Statistics",
                               join_field = "Grid_code",
                               fields = "Widthat100;Widthat75;Widthat43;Widthat16;Probability100;Probability75;Probability43;Probability16",
                               fm_option = "NOT_USE_FM",
                               field_mapping = None,
                               index_join_fields = "NO_INDEXES")
    
    # calculate streams with widths at different likelihoods
    arcpy.analysis.PairwiseBuffer(
    in_features = "memory/polyline",
    out_feature_class = "Polyline_PairwiseBuffer_100",
    buffer_distance_or_field = "Widthat100",
    dissolve_option = "NONE",
    dissolve_field = None,
    method = "PLANAR",
    max_deviation = "0 Meters")
    
    arcpy.analysis.PairwiseBuffer(
    in_features = "memory/polyline",
    out_feature_class = "Polyline_PairwiseBuffer_75",
    buffer_distance_or_field = "Widthat75",
    dissolve_option = "NONE",
    dissolve_field = None,
    method = "PLANAR",
    max_deviation = "0 Meters")
    
    arcpy.analysis.PairwiseBuffer(
    in_features = "memory/polyline",
    out_feature_class = "Polyline_PairwiseBuffer_43",
    buffer_distance_or_field = "Widthat43",
    dissolve_option = "NONE",
    dissolve_field = None,
    method = "PLANAR",
    max_deviation = "0 Meters")
    
    arcpy.analysis.PairwiseBuffer(
    in_features = "memory/polyline",
    out_feature_class = "Polyline_PairwiseBuffer_16",
    buffer_distance_or_field = "Widthat16",
    dissolve_option = "NONE",
    dissolve_field = None,
    method = "PLANAR",
    max_deviation = "0 Meters")
    
    arcpy.conversion.PolygonToRaster(
    in_features = "Polyline_PairwiseBuffer_100",
    value_field = "Probability100",
    out_rasterdataset = "Probability100",
    cell_assignment = "CELL_CENTER",
    priority_field = "NONE",
    cellsize = 25,
    build_rat =" BUILD")
    
    arcpy.conversion.PolygonToRaster(
    in_features = "Polyline_PairwiseBuffer_75",
    value_field = "Probability75",
    out_rasterdataset = "Probability75",
    cell_assignment = "CELL_CENTER",
    priority_field = "NONE",
    cellsize = 25,
    build_rat = "BUILD")
    
    arcpy.conversion.PolygonToRaster(
    in_features = "Polyline_PairwiseBuffer_43",
    value_field = "Probability43",
    out_rasterdataset = "Probability43",
    cell_assignment = "CELL_CENTER",
    priority_field = "NONE",
    cellsize = 25,
    build_rat = "BUILD")
    
    arcpy.conversion.PolygonToRaster(
    in_features = "Polyline_PairwiseBuffer_16",
    value_field = "Probability16",
    out_rasterdataset = "Probability16",
    cell_assignment = "CELL_CENTER",
    priority_field = "NONE",
    cellsize = 25,
    build_rat = "BUILD")

    # aggregate the different widths according to their likelihood
    arcpy.management.MosaicToNewRaster(
    input_rasters = "Probability100;Probability75;Probability43;Probability16",
    output_location = out_path,
    raster_dataset_name_with_extension = name,
    pixel_type = "64_BIT",
    cellsize = None,
    number_of_bands = 1,
    mosaic_method = "FIRST",
    mosaic_colormap_mode = "FIRST")

# ----------------------- AGGREGATION -----------------------
print('')
print("Aggregating results")
# delete the random DEM
Random_name = "Random_dhm25"
out_raster_stream = os.path.join(out_path,Random_name)
arcpy.management.Delete(out_raster_stream)
print("deleted")

# create a list of all remaining rasters
in_rasters = arcpy.ListRasters("Stream*", "All")
fcCount = len(in_rasters)
print(fcCount)

# aggregate all rasters and divide them by the number of runs
localsum = arcpy.sa.CellStatistics(in_rasters,"SUM","DATA")
print("calculating mean")
mean = localsum/fcCount
out_result = os.path.join(out_path,"mean_Stream")
mean.save(out_result)
print("completed")

In [ ]:
# Topographic Wetness Index Model
import arcpy
import os
import math
from arcpy.sa import *

# ----------------------- CONFIGURATION -----------------------
# set parallel cores
arcpy.env.addOutputsToMap = False
arcpy.env.parallelProcessingFactor = "100%"

# define digital elevation model
in_raster = r"DEM"

# configure environment settings
arcpy.env.extent = in_raster
arcpy.env.cellSize = in_raster
arcpy.env.overwriteOutput = True
    
# create describe object
desc = arcpy.Describe(arcpy.Describe(in_raster).path)
print(desc.path)

# create new file geodatabase
arcpy.management.CreateFileGDB(desc.path,"TWI.gdb")
out_path = os.path.join(desc.path,"TWI.gdb")
print(out_path)
arcpy.env.workspace = out_path

# ----------------------- MODELLING -----------------------
i = 200 #<-- number of iterations
for x in range(i):
    # create a new TWI
    TWI_name = "twi_"+str(x)
    print(TWI_name)
    out_raster_TWI = os.path.join(out_path,TWI_name)
    Random_Raster = arcpy.management.CreateRandomRaster(out_path = out_path,
                                                      out_name = "Random_DEM",
                                                      distribution = "NORMAL 0.0 1.5",
                                                      cellsize = 25,
                                                      build_rat = "DO_NOT_BUILD")

    Raster_Random_Raster = Raster(Random_Raster)
    Raster_in_raster = Raster(in_raster)
    output_raster = Raster_Random_Raster + Raster_in_raster

    FD = arcpy.sa.FlowDirection(in_surface_raster = output_raster,
                                force_flow = "NORMAL",
                                out_drop_raster = None,
                                flow_direction_type = "D8")
    
    out_accumulation_raster = arcpy.sa.FlowAccumulation(in_flow_direction_raster = FD,data_typ e= "FLOAT",flow_direction_type = "D8")

    SL = Slope(output_raster, output_measurement = "DEGREE")

    slope_radians = SL * (math.pi / 180)

    tan_slope = Tan(slope_radians)

    adjusted_flow_accumulation = out_accumulation_raster + 1

    TWI = Ln(adjusted_flow_accumulation / tan_slope)
    TWI.save(out_raster_TWI)

# ----------------------- AGGREGATION -----------------------
print('')
print("Aggregating results")
# delete Random Raster
arcpy.management.Delete(Random_Raster)
print("Random Raster deleted")

# create a list of all remaining rasters
in_rasters = arcpy.ListRasters("twi_*", "All")
fcCount = len(in_rasters)
print(fcCount)

# aggregate all rasters and divide them by the number of runs
localsum = arcpy.sa.CellStatistics(in_rasters,"SUM","DATA")
print("calculating mean")
mean = localsum/fcCount
out_result = os.path.join(out_path,"mean_TWI")
mean.save(out_result)
print("completed")

In [ ]:
# Bog Model
import arcpy
import os
from arcpy.sa import *

# ----------------------- CONFIGURATION -----------------------
# use parallel cores
arcpy.env.processorType = "CPU"
arcpy.env.addOutputsToMap = False
arcpy.env.parallelProcessingFactor = "100%"

# define digital elevation model
in_raster = r"DEM"

# configure environment settings
arcpy.env.extent = in_raster
arcpy.env.cellSize = in_raster
arcpy.env.overwriteOutput = True
arcpy.env.workspace = "memory"
    
# create describe object
desc = arcpy.Describe(arcpy.Describe(in_raster).path)
print(desc.path)

# create new file geodatabase
arcpy.management.CreateFileGDB(desc.path,"RF_Bog.gdb")
out_path = os.path.join(desc.path,"RF_Bog.gdb")
print(out_path)

# ----------------------- MODELLING -----------------------
i = 200 #<-- number of iterations
for x in range(i):
    # compile presence points    
    name = "Probability"+str(x)
    out_raster_pred = os.path.join(out_path,name)
    print(name)
    
    arcpy.management.CreateSpatialSamplingLocations(
        in_study_area = "Bog",
        out_features = r"Pos_points",
        sampling_method = "RANDOM",
        strata_id_field = None,
        strata_count_method = "EQUAL",
        bin_shape = "HEXAGON",
        bin_size = 100,
        h3_resolution = 7,
        num_samples = 200,
        num_samples_per_strata = 100,
        population_field = None,
        geometry_type = "POINT",
        min_distance = "200 Meters",
        spatial_relationship = "HAVE_THEIR_CENTER_IN")

    # compile absence points
    arcpy.management.CreateSpatialSamplingLocations(
        in_study_area = "No_bog",
        out_features = "Neg_points",
        sampling_method = "RANDOM",
        strata_id_field = None,
        strata_count_method = "EQUAL",
        bin_shape = "HEXAGON",
        bin_size = 100,
        h3_resolution = 7,
        num_samples = 200,
        num_samples_per_strata = 100,
        population_field = None,
        geometry_type = "POINT",
        min_distance = "200 Meters",
        spatial_relationship = "HAVE_THEIR_CENTER_IN")

    # merge the points
    arcpy.management.Merge(
        inputs = "Pos_points;Neg_points",
        output = "Points",
        field_mappings='CID "CID" true true false 4 Long 0 0,First,#,Pos_points,CID,-1,-1,Neg_points,CID,-1,-1',
        add_source = "ADD_SOURCE_INFO")

    # create a prediction surface
    arcpy.stats.Forest(
        prediction_type = "PREDICT_RASTER",
        in_features = "Points",
        variable_predict = "MERGE_SRC",
        treat_variable_as_categorical = "CATEGORICAL",
        explanatory_variables = None,
        distance_features = None,
        explanatory_rasters = "Flooding_Raster #;TWI #;Permeability true",
        features_to_predict = None,
        output_features = None,
        output_raster = out_raster_pred,
        explanatory_variable_matching = None,
        explanatory_distance_matching = None,
        explanatory_rasters_matching = "Flooding_Raster Flooding_Raster;TWI TWI;Permeability Permeability",
        output_trained_features = None,
        output_importance_table = None,
        use_raster_values = "TRUE",
        number_of_trees = 200,
        minimum_leaf_size = None,
        maximum_depth = None,
        sample_size = 100,
        random_variables = None,
        percentage_for_training = 20,
        output_classification_table = None,
        output_validation_table = None,
        compensate_sparse_categories = "FALSE",
        number_validation_runs = 10,
        calculate_uncertainty = "TRUE",
        output_trained_model = None,
        model_type = "FOREST-BASED",
        reg_lambda = None,
        gamma = None,
        eta = None,
        max_bins = None,
        optimize = "FALSE",
        optimize_algorithm = "",
        optimize_target = "",
        num_search = None,
        model_param_setting = None,
        output_param_tuning_table = None,
        include_probabilities = "HIGHEST_PROBABILITY_ONLY")

# ----------------------- Aggregation -----------------------
print('')
print("Aggregating results")
arcpy.env.workspace = out_path

# create a list of all remaining rasters
in_rasters = arcpy.ListRasters("Probability*","All")
fcCount = len(in_rasters)
print(fcCount)

# aggregate all rasters and divide them by the number of runs
localsum = arcpy.sa.CellStatistics(in_rasters,"SUM","DATA")
mean = localsum/fcCount
out_result = os.path.join(out_path,"mean")
mean.save(out_result)
print("finished")